# 01 — Ingestão e Construção da Base Analítica

**Objetivo:** Carregar a mensalizada, construir o target de turnover voluntário com janelas
forward-looking (3–12 meses), gerar rolling features por colaborador e indicadores contextuais
por grupo, e salvar as bases gold prontas para modelagem.

**Referência SPEC:** Seções 1-3 (Objetivo, Definição do Target, Estrutura da Base)

---

| Entrada | Descrição |
|---|---|
| `mensalizada_ficticio.parquet` | Base única — 1 linha = 1 colaborador × 1 mês |

| Saída | Descrição |
|---|---|
| `data/gold/base_tt_raw.parquet` | Treino/teste (com target) |
| `data/gold/base_val_raw.parquet` | Validação temporal (com target) |
| `data/gold/base_apl_raw.parquet` | Aplicação Set/2025+ (sem target) |

## 1 · Setup & Configuração

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"pandas {pd.__version__}  ·  numpy {np.__version__}")
print(f"Execução: {datetime.now():%Y-%m-%d %H:%M}")

pandas 2.3.3  ·  numpy 2.3.5
Execução: 2026-05-07 11:34


In [2]:
# ════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ════════════════════════════════════════════════════════════════
PROJECT_ROOT = Path.cwd().parent          # raiz do repo
DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_GOLD    = PROJECT_ROOT / "data" / "gold"

# Parâmetros do target
JANELA_PREDICAO    = 4          # janela principal usada no modelo (meses à frente)
JANELAS_TARGET     = list(range(3, 13))  # todas as janelas forward: 3m..12m
COL_TARGET         = f"fg_demitido_voluntario_{JANELA_PREDICAO}m"
COL_ID             = "id_colaborador"
COL_DATA           = "dt_mes_ano"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_RAW     : {DATA_RAW}")
print(f"DATA_GOLD    : {DATA_GOLD}")
print(f"Target       : {COL_TARGET}")
print(f"Janelas      : {JANELAS_TARGET}")

PROJECT_ROOT : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo
DATA_RAW     : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\raw
DATA_GOLD    : g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold
Target       : fg_demitido_voluntario_4m
Janelas      : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


## 2 · Criação da Estrutura de Pastas

In [3]:
# Estrutura de pastas do projeto (criada automaticamente)
DIRS = {
    "data/raw":                  "Bases brutas de entrada",
    "data/gold":                 "Base analítica unificada",
    "data/processed":            "Datasets de treino/teste/validação",
    "data/processed/splits":     "Splits treino/val/teste por grupo",
    "models":                    "Modelos serializados (.joblib)",
    "reports":                   "Relatórios e métricas",
    "reports/figures":           "Gráficos e visualizações",
    "reports/feature_selection": "Resultados de feature selection",
    "reports/shap":              "SHAP values e plots",
}

for subdir, desc in DIRS.items():
    p = PROJECT_ROOT / subdir
    p.mkdir(parents=True, exist_ok=True)

print("✓ Estrutura de pastas criada:")
for subdir, desc in DIRS.items():
    print(f"  {subdir:<35s} — {desc}")

✓ Estrutura de pastas criada:
  data/raw                            — Bases brutas de entrada
  data/gold                           — Base analítica unificada
  data/processed                      — Datasets de treino/teste/validação
  data/processed/splits               — Splits treino/val/teste por grupo
  models                              — Modelos serializados (.joblib)
  reports                             — Relatórios e métricas
  reports/figures                     — Gráficos e visualizações
  reports/feature_selection           — Resultados de feature selection
  reports/shap                        — SHAP values e plots


## 3 · Carga da Mensalizada

In [4]:
# ── Carregar mensalizada ──────────────────────────────────────
mensalizada = pd.read_parquet(DATA_RAW / "mensalizada_ficticio.parquet")
mensalizada[COL_DATA] = pd.to_datetime(mensalizada[COL_DATA])
mensalizada[COL_ID]   = mensalizada[COL_ID].astype(str)
mensalizada = mensalizada.sort_values([COL_ID, COL_DATA]).reset_index(drop=True)

print(f"mensalizada_ficticio : {mensalizada.shape[0]:,} linhas × {mensalizada.shape[1]} colunas")
print(f"  IDs únicos: {mensalizada[COL_ID].nunique():,}")
print(f"  Período   : {mensalizada[COL_DATA].min():%Y-%m} → {mensalizada[COL_DATA].max():%Y-%m}")
print(f"  Meses     : {mensalizada[COL_DATA].nunique()}")
print()
print("Colunas da mensalizada:")
for i, c in enumerate(mensalizada.columns, 1):
    print(f"  {i:3d}. {c:<50s} {str(mensalizada[c].dtype):<12s} nulls={mensalizada[c].isna().sum()}")

mensalizada_ficticio : 208,871 linhas × 69 colunas
  IDs únicos: 15,139
  Período   : 2024-01 → 2025-12
  Meses     : 24

Colunas da mensalizada:
    1. id_colaborador                                     object       nulls=0
    2. dt_mes_ano                                         datetime64[ns] nulls=0
    3. ds_cargo                                           object       nulls=0
    4. ds_uorg                                            object       nulls=0
    5. ds_centro_custo                                    object       nulls=0
    6. ds_genero                                          object       nulls=0
    7. ds_etnia                                           object       nulls=0
    8. ds_pcd_detalhe                                     object       nulls=20599
    9. ds_pcd                                             object       nulls=0
   10. ds_grade                                           object       nulls=0
   11. ds_tipo_turno                                      

In [5]:
# ════════════════════════════════════════════════════════════════
# CONSTRUÇÃO DO TARGET — janelas forward-looking (IMPL-9b)
#
# Para cada par (id, mês T) e janela N ∈ {3..12}:
#   fg_demitido_voluntario_Nm = 1   se fg_dem_vol==1 em T+1..T+N
#   fg_demitido_voluntario_Nm = NaN se fg_dem_inv==1 em T+1..T+N
#                                   OU janela ultrapassa o último mês do ID
#   fg_demitido_voluntario_Nm = 0   caso contrário
#
# Estratégia OTIMIZADA (sem transform-lambda em loop):
#   1. Extrair só as colunas de flag → matriz menor
#   2. Um único groupby().apply() por flag, retornando todas as janelas de uma vez
#   3. pd.concat ao final — sem atribuição coluna a coluna
# ════════════════════════════════════════════════════════════════

def _build_forward_max(series, janelas):
    """
    Para cada janela N em `janelas`, calcula o max de `series` em T+1..T+N
    usando reverse-rolling, sem lookahead.
    Retorna DataFrame com colunas '_fwd_{N}'.
    """
    s_rev = series.iloc[::-1]
    cols  = {}
    for N in janelas:
        # shift(-1) forward → inverter → rolling(N) → inverter de volta
        cols[f"_fwd_{N}"] = (
            s_rev.shift(1).rolling(N, min_periods=1).max().iloc[::-1].values
        )
    return pd.DataFrame(cols, index=series.index)


# ── Passo 1: last_mes por colaborador (para detectar janelas truncadas) ───
df = mensalizada.copy()
last_mes_map = df.groupby(COL_ID)[COL_DATA].max()
df["_last_mes"] = df[COL_ID].map(last_mes_map)

# ── Passo 2: rolling forward de fg_dem_vol e fg_dem_inv ──────────────────
print("Calculando forward-rolling de fg_dem_vol e fg_dem_inv...")

grp_vol = df.groupby(COL_ID, sort=False)["fg_dem_vol"]
grp_inv = df.groupby(COL_ID, sort=False)["fg_dem_inv"]

fwd_vol = grp_vol.apply(lambda s: _build_forward_max(s, JANELAS_TARGET)).droplevel(0)
fwd_inv = grp_inv.apply(lambda s: _build_forward_max(s, JANELAS_TARGET)).droplevel(0)

# Garantir alinhamento de índice
fwd_vol = fwd_vol.reindex(df.index)
fwd_inv = fwd_inv.reindex(df.index)

# ── Passo 3: montar colunas de target ────────────────────────────────────
target_cols_criadas = []
for N in JANELAS_TARGET:
    roll_vol = fwd_vol[f"_fwd_{N}"]
    roll_inv = fwd_inv[f"_fwd_{N}"]

    # Janela ultrapassa o último mês do colaborador?
    beyond = (df[COL_DATA] + pd.DateOffset(months=N)) > df["_last_mes"]

    col = f"fg_demitido_voluntario_{N}m"
    df[col] = np.where(
        roll_vol == 1,            1.0,
        np.where(
            (roll_inv == 1) | beyond, np.nan,
            0.0
        )
    )
    target_cols_criadas.append(col)

df.drop(columns=["_last_mes"], inplace=True)

print(f"✓ {len(target_cols_criadas)} colunas de target criadas: {target_cols_criadas}")
print()
for col in target_cols_criadas:
    n_1   = (df[col] == 1).sum()
    n_0   = (df[col] == 0).sum()
    n_nan = df[col].isna().sum()
    taxa  = n_1 / (n_1 + n_0) if (n_1 + n_0) > 0 else float("nan")
    print(f"  {col:<40s}  1={n_1:>6,}  0={n_0:>7,}  NaN={n_nan:>6,}  taxa={taxa:.2%}")

Calculando forward-rolling de fg_dem_vol e fg_dem_inv...
✓ 10 colunas de target criadas: ['fg_demitido_voluntario_3m', 'fg_demitido_voluntario_4m', 'fg_demitido_voluntario_5m', 'fg_demitido_voluntario_6m', 'fg_demitido_voluntario_7m', 'fg_demitido_voluntario_8m', 'fg_demitido_voluntario_9m', 'fg_demitido_voluntario_10m', 'fg_demitido_voluntario_11m', 'fg_demitido_voluntario_12m']

  fg_demitido_voluntario_3m                 1=13,842  0=153,906  NaN=41,123  taxa=8.25%
  fg_demitido_voluntario_4m                 1=17,310  0=139,625  NaN=51,936  taxa=11.03%
  fg_demitido_voluntario_5m                 1=20,404  0=126,380  NaN=62,087  taxa=13.90%
  fg_demitido_voluntario_6m                 1=23,139  0=114,089  NaN=71,643  taxa=16.86%
  fg_demitido_voluntario_7m                 1=25,541  0=102,631  NaN=80,699  taxa=19.93%
  fg_demitido_voluntario_8m                 1=27,705  0= 91,898  NaN=89,268  taxa=23.16%
  fg_demitido_voluntario_9m                 1=29,594  0= 81,969  NaN=97,308  taxa=2

In [6]:
## 4 · Rolling Features por Colaborador

#Para cada variável `vl_*` (exceto `vl_conjuge`, `vl_filhos`, `vl_dependentes`) são geradas
#agregações em janelas de **3 meses** e **6 meses** (mês atual + anteriores):
#média, mínimo, máximo e desvio padrão.

#**Estratégia otimizada:** groupby + rolling vetorizado sobre bloco de colunas de uma só vez,
#evitando loops python por coluna. Todas as 4 agregações x 2 janelas são calculadas em um
#único passe sobre os dados após o sort por `(id, mes)`.

In [7]:
# ════════════════════════════════════════════════════════════════
# ROLLING FEATURES POR COLABORADOR — 3m e 6m
#
# Colunas alvo: todas que começam com 'vl_', exceto
#               vl_conjuge, vl_filhos, vl_dependentes
#
# Estratégia otimizada:
#   - df já está ordenado por [COL_ID, COL_DATA] (sort feito na carga)
#   - groupby().rolling() opera sobre um bloco de colunas de uma vez
#     via pd.concat de resultados — sem transform-lambda em loop
#   - 2 janelas × 4 agregações em um único percurso por janela
# ════════════════════════════════════════════════════════════════

_VL_EXCLUIR = {"vl_conjuge", "vl_filhos", "vl_dependentes"}
JANELAS_ROLL_COLAB = [3, 6]

vl_cols = [
    c for c in df.columns
    if c.startswith("vl_") and c not in _VL_EXCLUIR
]

print(f"Colunas vl_* candidatas : {len(vl_cols)}")
for c in vl_cols:
    print(f"  {c}")
print()

# ── Bloco de colunas a agregar ────────────────────────────────
bloco = df[[COL_ID, COL_DATA] + vl_cols].copy()

rolling_frames = []   # lista de DataFrames parciais para pd.concat ao final

for W in JANELAS_ROLL_COLAB:
    print(f"Calculando rolling {W}m para {len(vl_cols)} colunas...", end=" ")

    # groupby().rolling() retorna MultiIndex (id, original_index) — flatten com reset
    grp_roll = (
        bloco[vl_cols]
        .groupby(df[COL_ID], sort=False)
        .rolling(window=W, min_periods=1)
    )

    mean_df = grp_roll.mean().droplevel(0).reindex(df.index)
    min_df  = grp_roll.min().droplevel(0).reindex(df.index)
    max_df  = grp_roll.max().droplevel(0).reindex(df.index)
    # std requer min_periods=2; janelas com só 1 valor → NaN (correto)
    std_df  = (
        bloco[vl_cols]
        .groupby(df[COL_ID], sort=False)
        .rolling(window=W, min_periods=2)
        .std()
        .droplevel(0)
        .reindex(df.index)
    )

    # Renomear colunas com sufixo
    mean_df.columns = [f"{c}_mean_{W}m" for c in vl_cols]
    min_df.columns  = [f"{c}_min_{W}m"  for c in vl_cols]
    max_df.columns  = [f"{c}_max_{W}m"  for c in vl_cols]
    std_df.columns  = [f"{c}_std_{W}m"  for c in vl_cols]

    rolling_frames.extend([mean_df, min_df, max_df, std_df])
    n_novas = len(vl_cols) * 4
    print(f"✓  ({n_novas} colunas × janela {W}m)")

# ── Concatenar todas as novas features ao df ──────────────────
df = pd.concat([df] + rolling_frames, axis=1)

total_novas = len(vl_cols) * 4 * len(JANELAS_ROLL_COLAB)
print(f"\n✓ {total_novas} colunas rolling adicionadas ao df")
print(f"  ({len(vl_cols)} vl_cols × 4 agg × {len(JANELAS_ROLL_COLAB)} janelas)")
print(f"  df shape: {df.shape[0]:,} × {df.shape[1]}")

Colunas vl_* candidatas : 44
  vl_tempo_funcao
  vl_salario
  vl_beneficio_tempo_1
  vl_beneficio_tempo_2
  vl_beneficio_extra_1
  vl_beneficio_extra_2
  vl_beneficio_extra_3
  vl_beneficio_extra_4
  vl_remuneracao_variavel
  vl_beneficio_extra_5
  vl_beneficio_extra_10
  vl_beneficio_extra_12
  vl_beneficio_extra_13
  vl_dias
  vl_dias_noturno
  vl_dias_diurno
  vl_dias_sabado
  vl_dias_domingo
  vl_horas_trabalhadas
  vl_dias_menos_6_horas
  vl_dias_mais_8_horas
  vl_horas_extras_dias
  vl_horas_extras_horas
  vl_horas_extras_extraordinarias_dias
  vl_horas_extras_extraordinarias_horas
  vl_adicional_noturno_dias
  vl_adicional_noturno_horas
  vl_atraso_dias
  vl_atraso_horas
  vl_falta_dias
  vl_falta_horas
  vl_economico_taxa_desemprego
  vl_economico_custo_vida
  vl_economico_salario_medio
  vl_economico_turnover_cnae_regiao
  vl_economico_pib_per_capita
  vl_tempo_empresa
  vl_distancia_trabalho_min
  vl_idade
  vl_abs_dias_sum_3m
  vl_abs_dias_sum_6m
  vl_abs_dias_sum_12m
  vl_s

## 5 · Features Derivadas Específicas

In [8]:
# ════════════════════════════════════════════════════════════════
# FEATURES DERIVADAS ESPECÍFICAS
#
# vl_perc_diurno   = vl_dias_diurno / vl_dias
# vl_perc_dias_he  = vl_horas_extras_dias / vl_dias
#
# Regras:
#   - Denominador == 0 → NaN (divisão segura)
#   - Denominador NaN  → NaN (propagação natural)
#   - Resultado clampado em [0, 1] (percentuais não podem ultrapassar 100%)
# ════════════════════════════════════════════════════════════════

features_derivadas = {
    "vl_perc_diurno":  ("vl_dias_diurno",       "vl_dias"),
    "vl_perc_dias_he": ("vl_horas_extras_dias",  "vl_dias"),
}

for nome, (numerador, denominador) in features_derivadas.items():
    if numerador not in df.columns:
        print(f"⚠ {nome}: numerador '{numerador}' não encontrado — pulando")
        continue
    if denominador not in df.columns:
        print(f"⚠ {nome}: denominador '{denominador}' não encontrado — pulando")
        continue

    denom_safe = df[denominador].replace(0, np.nan)
    df[nome]   = (df[numerador] / denom_safe).clip(0, 1)

    n_null = df[nome].isna().sum()
    print(f"  {nome:<25s}  média={df[nome].mean():.4f}  "
          f"min={df[nome].min():.4f}  max={df[nome].max():.4f}  "
          f"nulos={n_null:,}")

print(f"\n✓ {len(features_derivadas)} features derivadas criadas")
print(f"  df shape: {df.shape[0]:,} × {df.shape[1]}")

  vl_perc_diurno             média=0.8920  min=0.0000  max=1.0000  nulos=29,649
  vl_perc_dias_he            média=0.9351  min=0.0000  max=1.0000  nulos=29,649

✓ 2 features derivadas criadas
  df shape: 208,871 × 433


In [9]:
# ── Validação de duplicatas na mensalizada ────────────────────
dup = df.duplicated(subset=[COL_ID, COL_DATA]).sum()
if dup > 0:
    print(f"⚠ {dup} duplicatas (id, mês) encontradas — removendo (keep=last)")
    df = df.drop_duplicates(subset=[COL_ID, COL_DATA], keep="last").reset_index(drop=True)
else:
    print(f"✓ Sem duplicatas em (id_colaborador, dt_mes_ano)")

print(f"  Total: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  IDs únicos  : {df[COL_ID].nunique():,}")
print(f"  Período     : {df[COL_DATA].min():%Y-%m} → {df[COL_DATA].max():%Y-%m}")

✓ Sem duplicatas em (id_colaborador, dt_mes_ano)
  Total: 208,871 linhas × 433 colunas
  IDs únicos  : 15,139
  Período     : 2024-01 → 2025-12


## 5.5 · Indicadores Contextuais por Grupo (SPEC §3.4)

Para variáveis categóricas com alta cardinalidade (`ds_cargo`, `ds_uorg`, `ds_gestor`)
uma simples codificação one-hot geraria centenas de colunas esparsas e introduziria
risco de multicolinearidade. A alternativa adotada aqui é **quantificá-las por comportamento**:
calculamos, para cada grupo × mês, três indicadores baseados na população completa da mensalizada:

| Indicador | Definição |
|---|---|
| **Headcount** | Média do headcount mensal (count de `fg_ativo == 1`) nos **4 meses** (atual + 3 anteriores) |
| **Turnover voluntário** | Soma de `fg_dem_vol == 1` / soma de `fg_ativo == 1` nos mesmos 4 meses |
| **Turnover involuntário** | Soma de `fg_dem_inv == 1` / soma de `fg_ativo == 1` nos mesmos 4 meses |

A janela de 4 meses é calculada de forma **rolling** (sem lookahead), garantindo que
para qualquer registro em `dt_mes_ano = T`, apenas dados de `T`, `T-1`, `T-2`, `T-3`
sejam usados — mesmo que o grupo não exista em algum mês anterior (`min_periods=1`).

> Como `df` já é a mensalizada completa, não há join adicional — os indicadores são calculados
> diretamente sobre `df` e depois unidos de volta por `(dt_mes_ano, grupo)`.

In [10]:
# ════════════════════════════════════════════════════════════════
# INDICADORES CONTEXTUAIS POR GRUPO
# Janela deslizante de 4 meses sobre a população completa da mensalizada
# para quantificar variáveis categóricas de alta cardinalidade.
# ════════════════════════════════════════════════════════════════

JANELA_ROLL  = 4                     # mês atual + 3 anteriores
GRUPOS_ROLL  = [                     # (coluna, sufixo nos nomes das features)
    ("ds_cargo",  "cargo"),
    ("ds_uorg",   "uorg"),
    ("ds_gestor", "gestor"),
]
COL_ATIVO    = "fg_ativo"
COL_DEM_VOL  = "fg_dem_vol"
COL_DEM_INV  = "fg_dem_inv"


def calcular_indicadores_grupo(mens_df, group_col, group_suffix, col_data, n_meses=4):
    """
    Calcula headcount, turnover voluntário e involuntário com janela rolling
    de N meses por categoria, sem vazamento de dados futuros.

    Parâmetros
    ----------
    mens_df      : Mensalizada completa (toda a população, não apenas a amostra)
    group_col    : Coluna de agrupamento (ex.: 'ds_cargo')
    group_suffix : Sufixo para nomear as colunas de saída (ex.: 'cargo')
    col_data     : Nome da coluna de data (dt_mes_ano)
    n_meses      : Tamanho da janela rolling (padrão: 4)

    Retorna
    -------
    DataFrame com: [col_data, group_col, hc_Xm, tv_vol_Xm, tv_inv_Xm]
    """
    # Verificar colunas necessárias
    for c in [col_data, group_col, COL_ATIVO, COL_DEM_VOL, COL_DEM_INV]:
        if c not in mens_df.columns:
            raise ValueError(f"Coluna '{c}' não encontrada na mensalizada")

    # ── Passo 1: contagens mensais por grupo ─────────────────────
    monthly = (
        mens_df
        .groupby([col_data, group_col], observed=True, sort=True)
        .agg(
            cnt_ativo=(COL_ATIVO,   lambda x: (x == 1).sum()),
            cnt_vol  =(COL_DEM_VOL, lambda x: (x == 1).sum()),
            cnt_inv  =(COL_DEM_INV, lambda x: (x == 1).sum()),
        )
        .reset_index()
        .sort_values([group_col, col_data])
    )

    # ── Passo 2: rolling de N meses por grupo ────────────────────
    grp = monthly.groupby(group_col, sort=False)
    monthly["hc_roll"]   = grp["cnt_ativo"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).mean()
    )
    monthly["sum_ativo"] = grp["cnt_ativo"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).sum()
    )
    monthly["sum_vol"]   = grp["cnt_vol"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).sum()
    )
    monthly["sum_inv"]   = grp["cnt_inv"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).sum()
    )

    # ── Passo 3: taxas de turnover (sum_ativo como denominador) ──
    denom = monthly["sum_ativo"].replace(0, np.nan)
    monthly[f"vl_hc_{group_suffix}_{n_meses}m"]     = monthly["hc_roll"]
    monthly[f"vl_tv_vol_{group_suffix}_{n_meses}m"] = monthly["sum_vol"] / denom
    monthly[f"vl_tv_inv_{group_suffix}_{n_meses}m"] = monthly["sum_inv"] / denom

    feat_cols = [
        f"vl_hc_{group_suffix}_{n_meses}m",
        f"vl_tv_vol_{group_suffix}_{n_meses}m",
        f"vl_tv_inv_{group_suffix}_{n_meses}m",
    ]
    return monthly[[col_data, group_col] + feat_cols]


print("Função calcular_indicadores_grupo definida ✓")

Função calcular_indicadores_grupo definida ✓


In [11]:
# ── Aplicar para cada variável de grupo ───────────────────────
# df já é a mensalizada completa — passa df como fonte dos indicadores
stats_cols_added = []

for group_col, group_suffix in GRUPOS_ROLL:
    if group_col not in df.columns:
        print(f"⚠ {group_col} não encontrada no df — pulando")
        continue

    stats = calcular_indicadores_grupo(
        df, group_col, group_suffix, COL_DATA, JANELA_ROLL
    )

    new_cols = [c for c in stats.columns if c not in [COL_DATA, group_col]]

    # Remover colunas que já existam (idempotência em re-execução)
    df.drop(columns=[c for c in new_cols if c in df.columns], inplace=True, errors="ignore")

    n_antes = len(df)
    df = df.merge(stats, on=[COL_DATA, group_col], how="left")
    assert len(df) == n_antes, f"Merge alterou contagem! {n_antes} → {len(df)}"

    stats_cols_added.extend(new_cols)
    n_grupos = stats[group_col].nunique()
    for c in new_cols:
        null_pct = df[c].isna().mean() * 100
        print(f"  [{group_suffix}] {c}: média={df[c].mean():.4f} | nulos={null_pct:.1f}%")
    print(f"  ↳ {n_grupos} grupos '{group_col}' | {len(stats)} pares (mês × grupo)")

print(f"\n✓ {len(stats_cols_added)} colunas de contexto adicionadas:")
print(f"  {stats_cols_added}")
print(f"\nBase df após indicadores: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

  [cargo] vl_hc_cargo_4m: média=621.6521 | nulos=0.0%
  [cargo] vl_tv_vol_cargo_4m: média=0.0392 | nulos=0.0%
  [cargo] vl_tv_inv_cargo_4m: média=0.0134 | nulos=0.0%
  ↳ 101 grupos 'ds_cargo' | 2239 pares (mês × grupo)
  [uorg] vl_hc_uorg_4m: média=204.6460 | nulos=0.0%
  [uorg] vl_tv_vol_uorg_4m: média=0.0402 | nulos=0.0%
  [uorg] vl_tv_inv_uorg_4m: média=0.0136 | nulos=0.0%
  ↳ 90 grupos 'ds_uorg' | 1977 pares (mês × grupo)
  [gestor] vl_hc_gestor_4m: média=14.5386 | nulos=0.0%
  [gestor] vl_tv_vol_gestor_4m: média=0.0445 | nulos=0.3%
  [gestor] vl_tv_inv_gestor_4m: média=0.0146 | nulos=0.3%
  ↳ 1692 grupos 'ds_gestor' | 24400 pares (mês × grupo)

✓ 9 colunas de contexto adicionadas:
  ['vl_hc_cargo_4m', 'vl_tv_vol_cargo_4m', 'vl_tv_inv_cargo_4m', 'vl_hc_uorg_4m', 'vl_tv_vol_uorg_4m', 'vl_tv_inv_uorg_4m', 'vl_hc_gestor_4m', 'vl_tv_vol_gestor_4m', 'vl_tv_inv_gestor_4m']

Base df após indicadores: 208,871 linhas × 442 colunas


In [12]:
# ── 5.6 · Grupo Operacional (ds_grupo) ───────────────────────
# df já é a mensalizada — ds_grupo está presente diretamente.
# Apenas normalizar nomes antigos ("Área X") se necessário.

if "ds_grupo" not in df.columns:
    raise ValueError("Coluna 'ds_grupo' não encontrada na mensalizada. Verifique a base.")

_grp_map = {"Área 1": "Transporte", "Área 2": "Vendas", "Área 3": "Fábrica"}
df["ds_grupo"] = df["ds_grupo"].replace(_grp_map)

pct_null = df["ds_grupo"].isna().mean() * 100
print(f"ds_grupo presente no df: {df['ds_grupo'].nunique()} categorias")
print(f"  Nulos : {df['ds_grupo'].isna().sum():,}  ({pct_null:.1f}%)")
print(f"\nDistribuição por grupo:")
print(df["ds_grupo"].value_counts(dropna=False).to_string())

ds_grupo presente no df: 3 categorias
  Nulos : 0  (0.0%)

Distribuição por grupo:
ds_grupo
Transporte    101516
Vendas         80445
Fábrica        26910


## 6 · Validação do Target

In [13]:
# ── Verificar presença das colunas de target ──────────────────
target_cols = [c for c in df.columns if c.startswith("fg_demitido_voluntario_")]
print(f"Colunas de target encontradas: {target_cols}")

if COL_TARGET not in df.columns:
    candidates = [c for c in df.columns if c.startswith("fg_demitido_voluntario_")]
    if candidates:
        COL_TARGET = candidates[0]
        print(f"⚠ Target principal ajustado para: {COL_TARGET}")
    else:
        raise ValueError(
            f"Nenhuma coluna fg_demitido_voluntario_* encontrada. "
            f"Verifique se o bloco de construção do target executou corretamente."
        )

print(f"\nDistribuição do target ({COL_TARGET}):")
print(df[COL_TARGET].value_counts(dropna=False).to_string())
print(f"\nTaxa de turnover: {df[COL_TARGET].mean():.2%}")

Colunas de target encontradas: ['fg_demitido_voluntario_3m', 'fg_demitido_voluntario_4m', 'fg_demitido_voluntario_5m', 'fg_demitido_voluntario_6m', 'fg_demitido_voluntario_7m', 'fg_demitido_voluntario_8m', 'fg_demitido_voluntario_9m', 'fg_demitido_voluntario_10m', 'fg_demitido_voluntario_11m', 'fg_demitido_voluntario_12m']

Distribuição do target (fg_demitido_voluntario_4m):
fg_demitido_voluntario_4m
0.0    139625
NaN     51936
1.0     17310

Taxa de turnover: 11.03%


In [14]:

# ── fg_sem_output: identifica meses sem janela de predição válida ──
# Meses Set/2025+ não têm 4 meses de forward window dentro da base.
# Esses registros NÃO devem ser removidos pelo dropna — serão usados
# como base de aplicação (score sem target).
CUTOFF_SEM_OUTPUT = pd.Timestamp("2025-09-01")

df["fg_sem_output"] = (df[COL_DATA] >= CUTOFF_SEM_OUTPUT).astype(int)

n_com_output = (df["fg_sem_output"] == 0).sum()
n_sem_output = (df["fg_sem_output"] == 1).sum()

print(f"fg_sem_output aplicado:")
print(f"  fg_sem_output = 0 (com output):  {n_com_output:,} registros  ({df[COL_DATA][df['fg_sem_output']==0].min():%Y-%m} → {df[COL_DATA][df['fg_sem_output']==0].max():%Y-%m})")
print(f"  fg_sem_output = 1 (sem output):  {n_sem_output:,} registros  (Set/2025+)")
print(f"\nDistribuição do target ({COL_TARGET}) nos registros COM output (fg_sem_output=0):")
mask_com = df["fg_sem_output"] == 0
print(df.loc[mask_com, COL_TARGET].value_counts(dropna=False).to_string())
print(f"\nTaxa de turnover (registros com output, sem NaN de afastamento):")
target_com = df.loc[mask_com, COL_TARGET]
taxa = target_com.mean()
print(f"  {taxa:.2%}  (inclui NaN de afastamento/deslig. involuntário)")


fg_sem_output aplicado:
  fg_sem_output = 0 (com output):  171,467 registros  (2024-01 → 2025-08)
  fg_sem_output = 1 (sem output):  37,404 registros  (Set/2025+)

Distribuição do target (fg_demitido_voluntario_4m) nos registros COM output (fg_sem_output=0):
fg_demitido_voluntario_4m
0.0    139625
1.0     16133
NaN     15709

Taxa de turnover (registros com output, sem NaN de afastamento):
  10.36%  (inclui NaN de afastamento/deslig. involuntário)


## 7 · Validação de Qualidade

In [15]:
# ── Nulos por coluna ──────────────────────────────────────────
null_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
null_pct_pos = null_pct[null_pct > 0]

if len(null_pct_pos) > 0:
    print(f"Colunas com nulos ({len(null_pct_pos)}):")
    for col, pct in null_pct_pos.items():
        print(f"  {col:<45s} {pct:5.1f}%  ({df[col].isna().sum():,} linhas)")
else:
    print("✓ Nenhuma coluna com nulos")

print(f"\nTotal de nulos na base: {df.isna().sum().sum():,}")

Colunas com nulos (227):
  fg_demitido_voluntario_12m                     56.8%  (118,577 linhas)
  fg_demitido_voluntario_11m                     53.6%  (111,905 linhas)
  fg_demitido_voluntario_10m                     50.2%  (104,839 linhas)
  fg_demitido_voluntario_9m                      46.6%  (97,308 linhas)
  fg_demitido_voluntario_8m                      42.7%  (89,268 linhas)
  fg_demitido_voluntario_7m                      38.6%  (80,699 linhas)
  fg_demitido_voluntario_6m                      34.3%  (71,643 linhas)
  fg_demitido_voluntario_5m                      29.7%  (62,087 linhas)
  fg_demitido_voluntario_4m                      24.9%  (51,936 linhas)
  fg_demitido_voluntario_3m                      19.7%  (41,123 linhas)
  vl_perc_dias_he                                14.2%  (29,649 linhas)
  vl_perc_diurno                                 14.2%  (29,649 linhas)
  ds_pcd_detalhe                                  9.9%  (20,599 linhas)
  vl_horas_trabalhadas_std_3m       

In [16]:
# ── Tipos de dados ────────────────────────────────────────────
print("Resumo de tipos:")
print(df.dtypes.value_counts().to_string())

# ── Registros por mês ─────────────────────────────────────────
print(f"\nRegistros por mês:")
print(df.groupby(COL_DATA)[COL_ID].nunique().to_string())

Resumo de tipos:
float64           396
int64              24
object             17
Float64             5
datetime64[ns]      1

Registros por mês:
dt_mes_ano
2024-01-01    7746
2024-02-01    7564
2024-03-01    7643
2024-04-01    7769
2024-05-01    7861
2024-06-01    7931
2024-07-01    8080
2024-08-01    8230
2024-09-01    8457
2024-10-01    8779
2024-11-01    9037
2024-12-01    9246
2025-01-01    9312
2025-02-01    9016
2025-03-01    8989
2025-04-01    9083
2025-05-01    9140
2025-06-01    9165
2025-07-01    9206
2025-08-01    9213
2025-09-01    9298
2025-10-01    9301
2025-11-01    9355
2025-12-01    9450


In [17]:
# ── Taxa de turnover por grupo (se existir) ──────────────────
col_grupo = None
for c in ["grupo", "ds_grupo", "agrupamento_final"]:
    if c in df.columns:
        col_grupo = c
        break

if col_grupo:
    print(f"Taxa de turnover por {col_grupo}:")
    stats = df.groupby(col_grupo).agg(
        n_linhas=(COL_ID, "count"),
        n_ids=(COL_ID, "nunique"),
        taxa_turnover=(COL_TARGET, "mean"),
    )
    stats["taxa_turnover"] = stats["taxa_turnover"].map("{:.2%}".format)
    print(stats.to_string())
else:
    print("Coluna de grupo não encontrada — análise por grupo não disponível")

Taxa de turnover por ds_grupo:
            n_linhas  n_ids taxa_turnover
ds_grupo                                 
Fábrica        26910   1872        10.87%
Transporte    101516   7512        10.33%
Vendas         80445   5939        11.97%


In [ ]:

# ════════════════════════════════════════════════════════════════
# BASE FULL RAW — universo completo (sem filtros)
#
# Snapshot do `df` unificado APÓS todo o feature engineering
# (rolling stats por colaborador, indicadores de grupo, targets fwd)
# e ANTES dos filtros de fg_sem_output / dropna(target) / data.
#
# Uso EXCLUSIVO: notebook 02_1 (base_diagnostico) e análises ad-hoc.
# NÃO USAR no pipeline do modelo — para isso existem base_tt_raw,
# base_val_raw e base_apl_raw (geradas na próxima célula).
# ════════════════════════════════════════════════════════════════
out_full = DATA_GOLD / "base_full_raw.parquet"
df.to_parquet(out_full, index=False)

mb = out_full.stat().st_size / 1e6
print(f"✓ base_full_raw.parquet salva  (uso: diagnóstico/dashboard, NÃO modelo)")
print(f"  Shape   : {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  IDs     : {df[COL_ID].nunique():,}")
print(f"  Período : {df[COL_DATA].min():%Y-%m} → {df[COL_DATA].max():%Y-%m}")
print(f"  Tamanho : {mb:.1f} MB")
print(f"  Salvo em: {out_full}")


## 8 · Salvar Base Analítica

In [18]:

# ════════════════════════════════════════════════════════════════
# SALVAR TRÊS BASES  —  SPEC §0.6
#
#  base_tt_raw  : treino + teste  (Jan/2024–Mai/2025, target válido)
#  base_val_raw : validação       (Jun/2025–Ago/2025, target válido)
#  base_apl_raw : aplicação       (Set/2025+, fg_sem_output=1, sem target)
#
# Regras:
#   - tt e val: fg_sem_output==0, depois dropna(target)
#   - tt: dt_mes_ano <= Mai/2025
#   - val: Jun/2025 <= dt_mes_ano <= Ago/2025
#   - apl: fg_sem_output==1  (TODOS os registros, sem dropna)
# ════════════════════════════════════════════════════════════════

CUTOFF_TT  = pd.Timestamp("2025-05-31")   # último mês de treino/teste
CUTOFF_VAL_START = pd.Timestamp("2025-06-01")
CUTOFF_VAL_END   = pd.Timestamp("2025-08-31")

# ── base_tt_raw ───────────────────────────────────────────────
base_tt = (
    df[(df["fg_sem_output"] == 0) & (df[COL_DATA] <= CUTOFF_TT)]
    .dropna(subset=[COL_TARGET])
    .copy()
    .reset_index(drop=True)
)
out_tt = DATA_GOLD / "base_tt_raw.parquet"
base_tt.to_parquet(out_tt, index=False)

# ── base_val_raw ──────────────────────────────────────────────
base_val = (
    df[
        (df["fg_sem_output"] == 0) &
        (df[COL_DATA] >= CUTOFF_VAL_START) &
        (df[COL_DATA] <= CUTOFF_VAL_END)
    ]
    .dropna(subset=[COL_TARGET])
    .copy()
    .reset_index(drop=True)
)
out_val = DATA_GOLD / "base_val_raw.parquet"
base_val.to_parquet(out_val, index=False)

# ── base_apl_raw ──────────────────────────────────────────────
base_apl = (
    df[df["fg_sem_output"] == 1]
    .copy()
    .reset_index(drop=True)
)
out_apl = DATA_GOLD / "base_apl_raw.parquet"
base_apl.to_parquet(out_apl, index=False)

# ── Reportar ──────────────────────────────────────────────────
print(f"{'Base':<18} {'Shape':>16}  {'IDs':>8}  {'Período':>20}  {'Turnover':>10}")
print("-" * 80)
for nome, base, out_path in [
    ("base_tt_raw",  base_tt,  out_tt),
    ("base_val_raw", base_val, out_val),
    ("base_apl_raw", base_apl, out_apl),
]:
    n_ids  = base[COL_ID].nunique()
    per_st = base[COL_DATA].min().strftime("%Y-%m") if len(base) > 0 else "—"
    per_en = base[COL_DATA].max().strftime("%Y-%m") if len(base) > 0 else "—"
    taxa_s = f"{base[COL_TARGET].mean():.2%}" if nome != "base_apl_raw" else "NaN (sem target)"
    mb = out_path.stat().st_size / 1e6
    print(f"  {nome:<16} {str(base.shape):>16}  {n_ids:>8,}  {per_st} → {per_en}  {taxa_s:>14}  ({mb:.1f} MB)")
    print(f"  ✓ Salvo em: {out_path}")


Base                          Shape       IDs               Período    Turnover
--------------------------------------------------------------------------------
  base_tt_raw         (130846, 443)    11,611  2024-01 → 2025-05          10.73%  (148.8 MB)
  ✓ Salvo em: g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_tt_raw.parquet
  base_val_raw         (24912, 443)     8,828  2025-06 → 2025-08           8.41%  (32.9 MB)
  ✓ Salvo em: g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_val_raw.parquet
  base_apl_raw         (37404, 443)    10,273  2025-09 → 2025-12  NaN (sem target)  (47.8 MB)
  ✓ Salvo em: g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_apl_raw.parquet


In [19]:

# ── Resumo final ──────────────────────────────────────────────
print("="*70)
print("RESUMO — INGESTÃO")
print("="*70)
print(f"  Base unificada (antes do split) : {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  IDs únicos                      : {df[COL_ID].nunique():,}")
print(f"  Período completo                : {df[COL_DATA].min():%Y-%m} → {df[COL_DATA].max():%Y-%m}")
print()
print(f"  {'Base':<18} {'Linhas':>10}  {'IDs':>8}  {'Período':>20}  {'Turnover':>12}")
print("  " + "-"*72)
for nome, base in [
    ("base_tt_raw",  base_tt),
    ("base_val_raw", base_val),
    ("base_apl_raw", base_apl),
]:
    n_ids  = base[COL_ID].nunique()
    per_st = base[COL_DATA].min().strftime("%Y-%m") if len(base) > 0 else "—"
    per_en = base[COL_DATA].max().strftime("%Y-%m") if len(base) > 0 else "—"
    if nome != "base_apl_raw":
        taxa_s = f"{base[COL_TARGET].mean():.2%}"
    else:
        taxa_s = "NaN(sem target)"
    print(f"  {nome:<18} {len(base):>10,}  {n_ids:>8,}  {per_st} → {per_en}  {taxa_s:>14}")

print()
if "ds_grupo" in df.columns:
    print("  Taxa de turnover por grupo (base_tt_raw):")
    grupo_stats = base_tt.groupby("ds_grupo").agg(
        n_linhas=(COL_ID, "count"),
        n_ids=(COL_ID, "nunique"),
        taxa_turnover=(COL_TARGET, "mean"),
    )
    for g, row in grupo_stats.iterrows():
        print(f"    {g:<14}: {row['n_ids']:>4,} ids  |  {row['n_linhas']:>6,} reg  |  {row['taxa_turnover']:.2%}")
print("="*70)


RESUMO — INGESTÃO
  Base unificada (antes do split) : 208,871 linhas × 443 colunas
  IDs únicos                      : 15,139
  Período completo                : 2024-01 → 2025-12

  Base                   Linhas       IDs               Período      Turnover
  ------------------------------------------------------------------------
  base_tt_raw           130,846    11,611  2024-01 → 2025-05          10.73%
  base_val_raw           24,912     8,828  2025-06 → 2025-08           8.41%
  base_apl_raw           37,404    10,273  2025-09 → 2025-12  NaN(sem target)

  Taxa de turnover por grupo (base_tt_raw):
    Fábrica       : 1,494.0 ids  |  17,086.0 reg  |  10.87%
    Transporte    : 5,733.0 ids  |  63,351.0 reg  |  10.03%
    Vendas        : 4,499.0 ids  |  50,409.0 reg  |  11.56%
